In [1]:
!pip install os
!pip install glob
!pip install pandas


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# Convert date columns ('created_at, 'deadline', 'launched_at' to readable formats)

In [2]:
import os
import glob
import pandas as pd

In [3]:
# --- SETTINGS ---
INPUT_DIR = r"C:\Users\tansu\Downloads\Kickstarter_2026-01-12T09_37_51_016Z"
OUTPUT_DIR = r"C:\Users\tansu\Downloads\Kickstarter_readabledates"
os.makedirs(OUTPUT_DIR, exist_ok=True)

date_cols = ["created_at", "deadline", "launched_at"]


In [4]:
# Find all Kickstarter*.csv files in the input folder
pattern = os.path.join(INPUT_DIR, "Kickstarter*.csv")
files = sorted(glob.glob(pattern))

print(f"Found {len(files)} file(s). Example:", files[:3])

success, failed = [], []

for fp in files:
    try:
        df = pd.read_csv(fp)

        # Add readable date columns
        for col in date_cols:
            if col in df.columns:
                dt = pd.to_datetime(df[col], unit="s", errors="coerce")
                df[f"{col}_readable"] = dt.dt.strftime("%Y-%m-%d")
            else:
                # If a column is missing, still create it (keeps schema consistent)
                df[f"{col}_readable"] = pd.NA

        # Build output filename
        base = os.path.splitext(os.path.basename(fp))[0]  # e.g., Kickstarter001
        out_name = f"{base}_with_readable_dates.csv"
        out_path = os.path.join(OUTPUT_DIR, out_name)

        df.to_csv(out_path, index=False)
        success.append(base)

    except Exception as e:
        failed.append((os.path.basename(fp), str(e)))

print(f"\n✅ Done. Saved {len(success)} file(s) to:\n{OUTPUT_DIR}")

if failed:
    print(f"\n⚠️ Failed on {len(failed)} file(s):")
    for fname, err in failed[:10]:  # show first 10 errors
        print("-", fname, "->", err)


✅ Done. Saved 84 file(s) to:
C:\Users\tansu\Downloads\Kickstarter_readabledates


# Merge all 84 csv files 

In [5]:
import os
import glob
import pandas as pd

INPUT_DIR = r"C:\Users\tansu\Downloads\Kickstarter_readabledates"
OUTPUT_FILE = r"C:\Users\tansu\Downloads\Kickstarter_readabledates\Kickstarter_ALL.csv"

files = sorted(glob.glob(os.path.join(INPUT_DIR, "Kickstarter*_with_readable_dates.csv")))

print(f"Found {len(files)} files")

df_list = []

for fp in files:
    df = pd.read_csv(fp)

    # Add source file column (VERY useful)
    df["source_file"] = os.path.basename(fp)

    df_list.append(df)

# Concatenate all files
df_all = pd.concat(df_list, ignore_index=True)

print("Total rows:", df_all.shape[0])

# Save merged dataset
df_all.to_csv(OUTPUT_FILE, index=False)

print("✅ Merged CSV saved as:", OUTPUT_FILE)


✅ Merged CSV saved as: C:\Users\tansu\Downloads\Kickstarter_readabledates\Kickstarter_ALL.csv


# Remove rows with duplicated info for blurb, created_at, deadline, launched_at

In [7]:
import pandas as pd

path = r"C:\Users\tansu\Downloads\Kickstarter_readabledates\Kickstarter_ALL.csv"
df = pd.read_csv(path)

print("Original rows:", len(df))

subset_cols = ["blurb", "created_at", "deadline", "launched_at"]

# Count duplicates based on selected columns
n_dupes = df.duplicated(subset=subset_cols).sum()
print("Rows duplicated on blurb + created_at + deadline + launched_at:", n_dupes)

Original rows: 266279
Rows duplicated on blurb + created_at + deadline + launched_at: 59405


In [8]:
df_clean = df.drop_duplicates(subset=subset_cols, keep="first")

print("Rows after deduplication:", len(df_clean))
print("Rows removed:", len(df) - len(df_clean))

Rows after deduplication: 206874
Rows removed: 59405


In [9]:
out_path = r"C:\Users\tansu\Downloads\Kickstarter_readabledates\Kickstarter_ALL_deduped_blurb_dates.csv"
df_clean.to_csv(out_path, index=False)

print("✅ Cleaned file saved to:", out_path)

✅ Cleaned file saved to: C:\Users\tansu\Downloads\Kickstarter_readabledates\Kickstarter_ALL_deduped_blurb_dates.csv


In [10]:
#Check no duplicates remain
df_clean.duplicated(subset=subset_cols).sum()

np.int64(0)

In [12]:
#Inspect examples of removed duplicates (before deletion)
df[df.duplicated(subset=subset_cols, keep=False)] \
  .sort_values(subset_cols) \
  .head(100)

,backers_count,blurb,category,converted_pledged_amount,country,country_displayable_name,created_at,creator,currency,currency_symbol,...,static_usd_rate,urls,usd_exchange_rate,usd_pledged,usd_type,video,created_at_readable,deadline_readable,launched_at_readable,source_file
108618,4,""" Podcast For The ""Struggling Millennial.""","{""id"":358,""name"":""Photo"",""analytics_name"":""Pho...",26.0,US,the United States,1570794282,"{""id"":1682007130,""name"":""Vivi F"",""slug"":""notpe...",USD,$,...,1.000000,"{""web"":{""project"":""https://www.kickstarter.com...",1.000000,26.00000,domestic,NaN,2019-10-11,2020-03-01,2020-01-01,Kickstarter035_with_readable_dates.csv
142633,4,""" Podcast For The ""Struggling Millennial.""","{""id"":358,""name"":""Photo"",""analytics_name"":""Pho...",26.0,US,the United States,1570794282,"{""id"":1682007130,""name"":""Vivi F"",""slug"":""notpe...",USD,$,...,1.000000,"{""web"":{""project"":""https://www.kickstarter.com...",1.000000,26.00000,domestic,NaN,2019-10-11,2020-03-01,2020-01-01,Kickstarter045_with_readable_dates.csv
185857,374,""" The ultimate Dark Star! "" Help reissue this ...","{""id"":43,""name"":""Rock"",""analytics_name"":""Rock""...",46212.0,US,the United States,1715188909,"{""id"":1983757235,""name"":""Important Records"",""s...",USD,$,...,1.000000,"{""web"":{""project"":""https://www.kickstarter.com...",1.000000,46212.00000,domestic,"{""id"":1295801,""status"":""successful"",""hls"":""htt...",2024-05-08,2024-11-30,2024-10-31,Kickstarter059_with_readable_dates.csv
228642,374,""" The ultimate Dark Star! "" Help reissue this ...","{""id"":43,""name"":""Rock"",""analytics_name"":""Rock""...",46212.0,US,the United States,1715188909,"{""id"":1983757235,""name"":""Important Records"",""s...",USD,$,...,1.000000,"{""web"":{""project"":""https://www.kickstarter.com...",1.000000,46212.00000,domestic,"{""id"":1295801,""status"":""successful"",""hls"":""htt...",2024-05-08,2024-11-30,2024-10-31,Kickstarter072_with_readable_dates.csv
35096,121,"""(Luna)tics"" is a collection of work being pre...","{""id"":254,""name"":""Performances"",""analytics_nam...",9161.0,US,the United States,1427071762,"{""id"":316713633,""name"":""Fuerta Dance"",""is_regi...",USD,$,...,1.000000,"{""web"":{""project"":""https://www.kickstarter.com...",1.000000,9161.00000,domestic,"{""id"":515148,""status"":""successful"",""hls"":null,...",2015-03-23,2015-04-22,2015-03-23,Kickstarter012_with_readable_dates.csv
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8493,25,"""Badass Sexy Girls"" Vol 1- "" The new Artbook b...","{""id"":252,""name"":""Graphic Novels"",""analytics_n...",548.0,IT,Italy,1740295592,"{""id"":1530752856,""name"":""pink-liza"",""is_regist...",EUR,€,...,1.118099,"{""web"":{""project"":""https://www.kickstarter.com...",1.141198,537.80573,domestic,NaN,2025-02-23,2025-06-09,2025-05-20,Kickstarter003_with_readable_dates.csv
182738,25,"""Badass Sexy Girls"" Vol 1- "" The new Artbook b...","{""id"":252,""name"":""Graphic Novels"",""analytics_n...",548.0,IT,Italy,1740295592,"{""id"":1530752856,""name"":""pink-liza"",""is_regist...",EUR,€,...,1.118099,"{""web"":{""project"":""https://www.kickstarter.com...",1.141198,537.80573,domestic,NaN,2025-02-23,2025-06-09,2025-05-20,Kickstarter058_with_readable_dates.csv
8583,40,"""Beast of a Woman"" - new work by Meegan Herten...","{""id"":254,""name"":""Performances"",""analytics_nam...",3765.0,US,the United States,1425436932,"{""id"":2130741589,""name"":""Meegan Hertensteiner""...",USD,$,...,1.000000,"{""web"":{""project"":""https://www.kickstarter.com...",1.000000,3765.00000,domestic,"{""id"":508848,""status"":""successful"",""hls"":null,...",2015-03-04,2015-03-30,2015-03-09,Kickstarter003_with_readable_dates.csv
40836,40,"""Beast of a Woman"" - new work by Meegan Herten...","{""id"":254,""name"":""Performances"",""analytics_nam...",3765.0,US,the United States,1425436932,"{""id"":2130741589,""name"":""Meegan Hertensteiner""...",USD,$,...,1

# flatten 'creator' column to get creator_id, creator_name, creator_backing_action_count

In [13]:
import pandas as pd
import json


In [14]:
path = r"C:\Users\tansu\Downloads\Kickstarter_readabledates\Kickstarter_ALL_deduped_blurb_dates.csv"
df = pd.read_csv(path)

print("Rows loaded:", len(df))


Rows loaded: 206874


In [15]:
# Define a safe parser for the creator JSON
def parse_creator(x):
    try:
        c = json.loads(x) if pd.notna(x) else {}
        return pd.Series({
            "creator_id": c.get("id"),
            "creator_name": c.get("name"),
            "creator_is_superbacker": c.get("is_superbacker"),
            "creator_backing_action_count": c.get("backing_action_count")
        })
    except Exception:
        return pd.Series({
            "creator_id": None,
            "creator_name": None,
            "creator_is_superbacker": None,
            "creator_backing_action_count": None
        })


In [16]:
# Apply parser and merge back to dataframe
creator_cols = df["creator"].apply(parse_creator)

df = pd.concat([df, creator_cols], axis=1)


In [17]:
#Quick sanity checks
df[[
    "creator_id",
    "creator_name",
    "creator_is_superbacker",
    "creator_backing_action_count"
]].head()


,creator_id,creator_name,creator_is_superbacker,creator_backing_action_count
0,1.228915e+09,Skyler Husebye,None,0.0
1,1.665133e+09,FANG FANG,None,0.0
2,1.183234e+09,Brendan Cleaves & Stuart Foreman,None,0.0
3,1.379754e+09,Mixed Doubles,None,0.0
4,3.468632e+08,Cleveland High School PTA,None,0.0


In [18]:
# And check missingness:
df[[
    "creator_id",
    "creator_is_superbacker",
    "creator_backing_action_count"
]].isna().mean()


creator_id                      0.002277
creator_is_superbacker          1.000000
creator_backing_action_count    0.002277
dtype: float64

In [20]:
df = df.drop(columns=["creator_is_superbacker"])
out_path = r"C:\Users\tansu\Downloads\Kickstarter_readabledates\Kickstarter_ALL_flattened_creator.csv"
df.to_csv(out_path, index=False)

print("✅ Creator column flattened and file saved.")


✅ Creator column flattened and file saved.


# Flatten location column to get location_id, location_city, location_country, location_state

In [21]:
import pandas as pd
import json


In [22]:
path = r"C:\Users\tansu\Downloads\Kickstarter_readabledates\Kickstarter_ALL_flattened_creator.csv"
df = pd.read_csv(path)

print("Rows loaded:", len(df))


Rows loaded: 206874


In [24]:
# Define safe location parser
def parse_location(x):
    try:
        loc = json.loads(x) if pd.notna(x) else {}
        return pd.Series({
            "location_id": loc.get("id"),
            "location_city": loc.get("displayable_name"),
            "location_country": loc.get("country"),
            "location_state": loc.get("state")
        })
    except Exception:
        return pd.Series({
            "location_id": None,
            "location_city": None,
            "location_country": None,
            "location_state": None
        })


In [26]:
# Apply parser and merge back
location_cols = df["location"].apply(parse_location)
df = pd.concat([df, location_cols], axis=1)


In [27]:
#Sanity checks
df[[
    "location_id",
    "location_city",
    "location_country",
    "location_state"
]].head()

,location_id,location_city,location_country,location_state
0,2402292.0,"Fargo, ND",US,ND
1,2471217.0,"Philadelphia, PA",US,PA
2,44418.0,"London, UK",GB,England
3,44418.0,"London, UK",GB,England
4,2475687.0,"Portland, OR",US,OR


In [28]:
#
df[[
    "location_id",
    "location_country"
]].isna().mean()


location_id         0.001392
location_country    0.001392
dtype: float64

In [29]:
out_path = r"C:\Users\tansu\Downloads\Kickstarter_readabledates\Kickstarter_ALL_flattened_creator_location.csv"
df.to_csv(out_path, index=False)

print("✅ Location column flattened and file saved.")


✅ Location column flattened and file saved.


# profile_has_feature_image = 1 only when the profile explicitly shows a feature image, and 0 otherwise.
if either of the following is true in the profile JSON:
show_feature_image
should_show_feature_image_section

In [30]:
import pandas as pd
import json


In [31]:
path = r"C:\Users\tansu\Downloads\Kickstarter_readabledates\Kickstarter_ALL_flattened_creator_location.csv"
df = pd.read_csv(path)

print("Rows loaded:", len(df))


Rows loaded: 206874


In [32]:
def parse_profile_feature(x):
    try:
        p = json.loads(x) if pd.notna(x) else {}
        return int(
            p.get("show_feature_image") is True
            or p.get("should_show_feature_image_section") is True
        )
    except Exception:
        return 0


In [34]:
df["profile_has_feature_image"] = df["profile"].apply(parse_profile_feature)


In [35]:
out_path = r"C:\Users\tansu\Downloads\Kickstarter_readabledates\Kickstarter_ALL_final.csv"
df.to_csv(out_path, index=False)

print("✅ profile_has_feature_image created and file saved.")


✅ profile_has_feature_image created and file saved.


# Reorder and drop columns

In [1]:
import pandas as pd

# Load dataset
path = r"C:\Users\tansu\Downloads\Kickstarter_readabledates\Kickstarter_ALL_final.csv"
df = pd.read_csv(path)

# Final column order (updated)
final_cols = [
    # Campaign outcomes & performance
    "backers_count",
    "usd_pledged",
    "pledged",
    "goal",
    "percent_funded",
    "state",
    "spotlight",
    "staff_pick",
    "is_in_post_campaign_pledging_phase",

    # Time & lifecycle
    "created_at",
    "created_at_readable",
    "launched_at",
    "launched_at_readable",
    "deadline",
    "deadline_readable",
    "is_launched",
    "prelaunch_activated",
    "state_changed_at",

    # Creator characteristics
    "creator_id",
    "creator_name",
    "creator_backing_action_count",

    # Geography
    "location_country",
    "location_state",
    "location_city",
    "country",
    "country_displayable_name",

    # Project content & signals
    "name",
    "blurb",
    "category",
    "video",
    "profile_has_feature_image",
    "is_starrable",

    # Currency & normalization
    "currency",
    "current_currency",
    "fx_rate",
    "usd_exchange_rate",
    "static_usd_rate",
    "usd_type",
    "converted_pledged_amount",

    # Identifiers & provenance
    "id",
    "slug",
    "url",
    "source_file"
]

# Keep only columns that exist (robust to small schema differences)
final_cols = [c for c in final_cols if c in df.columns]

df_final = df[final_cols]

print("Final shape:", df_final.shape)


Final shape: (206874, 42)


In [2]:
out_path = r"C:\Users\tansu\Downloads\Kickstarter_readabledates\Kickstarter_ALL_analysis_ready.csv"
df_final.to_csv(out_path, index=False)

print("✅ Final dataset (with is_starrable & is_launched) saved.")


✅ Final dataset (with is_starrable & is_launched) saved.
